# Data Merging — Stage 5 Themes 01: Reconcile

## Input
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/{7 table names}.parquet` (from Stage 4 notebook 01)
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/{agg_means, agg_full_moments, panel}.parquet` (from Stage 4 notebooks 02/03)
- `Data/Splits/themes/combined_means_theme_assignment.csv` — the *corrected* old theme assignment file (explicitly not the buggy copy under `Stage_3_Model_Ready/themes/`)
- `lib.review` (`base_factor_map`), `lib.config` (`OUT`)

## Purpose
The first notebook in Stage 5 Themes. Its job is entirely reconciliation and bootstrapping: it determines the cadence (daily/weekly/monthly) and level (stock/macro) of every base factor surviving into the final model-ready datasets, matches each factor against a previously-built theme/subtheme classification from an earlier version of the pipeline, identifies which factors are genuinely new (need manual classification) versus genuinely dropped, and flags structural problems (mixed-cadence subthemes, undersized subthemes) that must be resolved before notebook 02 can hard-code the final theme dictionary. Notebook 02 is described as reading nothing from `Stage_3_Model_Ready` and hard-coding its theme dictionary directly — so this notebook exists specifically to produce the analysis needed to write that dictionary correctly, after which the old `Stage_3_Model_Ready/themes` directory can be deleted entirely.

## Cell 1 — Determine Cadence and Level Per Base Factor
Neither cadence nor level can be reliably recovered from a column name alone (`copper_monthly` happens to say so, `bank_credit` does not). Instead, both are read directly from which of the seven `01_unioned/` source tables each factor actually appears in — since those tables are already split by source frequency, this is exact rather than inferred.

- **Level (stock vs. macro):** a base factor counts as stock-level if and only if it has a `_cwmean` column in one of the two full-moments tables (`agg_market_daily_full_moments`, `agg_market_monthly_full_moments`). Everything else appearing in a means table is macro. Because the means tables use *bare* base-factor names (`Tax`, not `Tax_cwmean`), this requires cross-referencing against the full-moments tables rather than reading anything off the means-table column names directly.
- **Cadence:** determined by which of the three means-level source tables (`agg_market_daily_means`, `weekly_raw`, `agg_market_monthly_means`) a factor's column appears in.
- **Conflict detection:** explicitly checks for any base factor appearing at more than one cadence, which would indicate an inconsistency. This check exists because of a known historical problem: the old pipeline discovered monthly `VolumeTrend` sitting inside a daily subtheme, and weekly claims inside a monthly subtheme, and had to patch both after the fact. Recording cadence explicitly per factor here is meant to convert "one cadence per subtheme" from a manual patch into an assertion that notebook 02 can enforce automatically.

Reports the count of base factors catalogued, breakdown by cadence, breakdown by level, and any conflicts found.

## Cell 2 — Base Factors Present in the Final Datasets
Loads the column set from each of the three final assembled tables (`agg_means`, `agg_full_moments`, `panel`), maps each column to its base factor via `rv.base_factor_map`, and takes the union across all three to get `new_base` — the full set of base factors that actually need a theme assignment.

Cross-checks that every factor in `new_base` also appears in the cadence/level catalogue built in Cell 1 (flagging any that don't, which would indicate a factor slipped through without being recorded). Also verifies the expected structural relationship between the panel and `agg_means`: since the Stage 4 union step enforced an identical stock-level base factor set across pipelines, the panel's base factors should be a clean subset of `agg_means`'s (panel = stock + macro; the aggregate side adds only *moments*, which aren't base factors in their own right). Any asymmetry here is printed explicitly rather than assumed away.

## Cell 3 — Name Reuse Across Datasets
Documents an important and easily-confusable subtlety: the *same column name* means a genuinely different quantity depending on which dataset it appears in. `Tax` in the panel is one individual stock's value; `Tax` in `agg_means` is the cap-weighted *market average* of that same underlying quantity; `Tax_cwmean` in `agg_full_moments` is the identical quantity to the second case, just under a different name. This is explicitly not treated as a bug — it's exactly why an earlier assembly notebook's collision assertion fired when the panel first attempted to pull in the aggregate's means (Stage 4 notebook 03), and why the panel deliberately takes only macro features from the aggregate rather than the stock-level cwmeans.

This matters directly for the theme file's design: a single theme assignment keyed on **base factor** can correctly serve all three datasets simultaneously, with the `level` column recording which interpretation (individual stock value vs. cross-sectional market average) applies in each case.

Reports counts of names shared between panel and `agg_means`, split by whether they're stock-level or macro-level, with examples of each.

Also runs a consistency check: every stock-level base factor identified in the means table should have all five moment suffixes present in the full-moments table — any that don't are flagged as "missing moments," with the expected causes noted explicitly (Stage 2's `drop_undefined` removing degenerate skew/kurt columns, and Stage 3's Rule 2/Rule 6 removing individual moments from ~28 factors whose `cwmean` itself survived).

## Cell 4 — Match Against the Old Theme Assignment
Loads the old theme assignment from `Data/Splits/themes/combined_means_theme_assignment.csv` — explicitly **not** the copy under `Stage_3_Model_Ready/themes/`, because that older copy has a known float-truncation bug: subtheme ID `'1.10'` was written to CSV and read back by pandas as the float `1.1`, silently colliding with the genuinely-distinct subtheme `'1.1'`. Six subthemes were affected this way (1.10, 2.10, 3.10, 8.10, 9.10, 12.10), which is why a mapping that should contain 107 subthemes reads back as only 101 from the buggy file. The `Data/Splits/themes/` copy already has this fixed and uses underscore-separated IDs (`1_10` instead of `1.10`), and additionally already contains two mixed-cadence subtheme splits (`2_11`, `12_11`) that the old pipeline had to retrofit — so both known cadence conflicts should come back clean when checked against this corrected file.

**Normalisation applied:** only the `monthly_` prefix is stripped from old column names (`norm()`), and explicitly *nothing else* — in particular, `_spread` is deliberately **not** stripped as a moment suffix here, because the old file is a *means-only* assignment where every name is already a bare base factor, and thirteen genuine base factors have names that legitimately end in `_spread` (`bbb_aaa_spread`, `bid_ask_spread`, `bull_bear_spread`, `vix_term_spread`, etc.) — stripping it would corrupt those names and cause each to spuriously appear as both "new" and "dropped" simultaneously.

After normalisation:
- Checks subtheme count is close to the expected ~107 (flagging if lower, which would indicate the float-truncation bug persists).
- Checks for any name collisions introduced by the `monthly_`-stripping normalisation itself.
- Splits `new_base` into three categories relative to the old assignment: **matched** (inherit their old theme directly), **missing/NEW** (need manual classification), and **gone/dropped** (were classified before, no longer present).
- **Near-match detection:** for every "new" factor, checks whether its name is a prefix/suffix match against any "dropped" factor's name — catching cases where a factor is probably the *same* factor under a renamed column, meaning the normalisation logic still has a gap, rather than being genuinely new.
- Prints the full "new factors needing classification" table with their cadence and level.
- Prints the "dropped" factors grouped by their old subtheme, with an explicit expectation set in a comment: 85 union drops, 12 high-imputation manual drops, 8 calendar factors (subtheme 14_1), 5 Fed balance-sheet levels replaced by their weekly `_diff` versions, plus any earlier Stage 1.5/2 exclusions that never made it into the new pipeline at all.
- **Emptied subthemes** — flags any old subtheme with zero surviving members, distinguishing a genuine deletion (subtheme 14_1 "Calendar," all 8 features excluded in Stage 3) from what's actually a *rename*/restructuring (subtheme 9_7 "Fed Balance Sheet," whose five level factors were replaced by weekly-differenced versions in the Stage 3 rebuild, four of which survive under new names).

## Cell 5 — Build the Starter Assignment and Check Structural Constraints
Builds `starter` — one row per base factor in `new_base`, carrying cadence, level, and (where matched) the inherited old theme/subtheme, with a `status` of `'matched'` or `'NEEDS CLASSIFYING'`. Saves this directly as the file notebook 02 will presumably read from or be informed by when hard-coding its final theme dictionary.

Then runs two structural checks on the matched subset:

- **One cadence per subtheme (a hard requirement).** The sparse KAN's update mask requires a subtheme to fire on the specific days its features actually change; a subtheme mixing daily and monthly features has no single well-defined update day. Since the corrected source file already carries the two known mixed-cadence splits (2_11, 12_11), this check is expected to come back clean — anything flagged here represents a *new* conflict introduced since the old file was built, not a known and already-handled one.
- **Stock/macro mixing within a subtheme** — checked and reported, but explicitly labelled informational rather than a problem requiring action.

## Cell 6 — Subtheme Sizes and Merge Candidates
Computes subtheme sizes (member count) across the matched factors, with an explicit target: a subtheme with only one member is "a passthrough edge, not a group" — it gives the sparse KAN's grouped structure no advantage over a fully dense model, so subthemes of size 1 or 2 are flagged as merge candidates (2 is "tolerated where no sensible sibling exists").

Merge candidates are searched **only within the same theme and the same cadence** — explicitly justified because a subtheme's cadence determines its update day, so a daily orphan subtheme cannot be merged into a monthly one regardless of thematic similarity. For each undersized subtheme, prints its members and any same-theme, same-cadence sibling subthemes it could plausibly be merged into (or a note that no such sibling exists, meaning it must either stay standalone or be reassigned to a different theme entirely).

Finishes with a per-theme summary: subtheme count, total factor count, and minimum subtheme size per theme.

## Diagnostic Cell (Standalone) — Schema Comparison Against the Old Split Files
A final, disconnected sanity check comparing the column schema of the old pipeline's test-split file (`Data/Splits/Split_A/full_moments_test.parquet`) against the new pipeline's equivalent (`Stage_5_Model_Ready/04_splits/Split_A/agg_full_moments_test.parquet`), reading via `pyarrow.parquet.read_schema` for exact column names. Notes the meta-column differences between old and new (the old file's target naming — `minret_5d`, `minret_5d_z` — versus the new file's `minret_5d_pct`) so the reported feature counts can be adjusted for the different number of meta columns in each.

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/starter_assignment.csv` — one row per base factor in the final datasets, with cadence, level, inherited theme/subtheme (where matched), and classification status. This is the hand-off artifact notebook 02 uses to finalize its hard-coded theme dictionary.

All other output in this notebook is printed for inspection/manual decision-making rather than saved to file.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append('../..')
import lib.review as rv
from lib.config import OUT

UNI_DIR = OUT / '01_unioned'
ASM_DIR = OUT / '02_assembled'
OUT_DIR = OUT / '05_themes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Read ONCE, to bootstrap. Notebook 02 hard-codes its dictionary and reads
# nothing from Stage_3_Model_Ready, so that directory can then be deleted.
OLD = Path('../../../Data/Data_Collection/Final/Stage_3_Model_Ready/themes'
           '/combined_means_theme_assignment.csv')

MOMENTS = ('_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread')

FINAL_META = {
    'agg_means':        ['date', 'target_daily_return'],
    'agg_full_moments': ['date', 'target_daily_return'],
    'panel':            ['permno', 'date', 'dlyret', 'dlycap'],
}

pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 500)

# ── CADENCE AND LEVEL, read from the source tables ───────────────────────────
# Neither is recoverable from a column name. `copper_monthly` happens to say so;
# `bank_credit` does not. The seven files in 01_unioned/ are already split by
# source table, so both are exact.
#
# LEVEL: a stock factor is exactly one with a _cwmean column in a full-moments
# table. Everything else in the means table is macro. The means tables use BARE
# base factor names -- `Tax`, not `Tax_cwmean` -- so the identification is a
# cross-reference between the two aggregate tables, not a read of the name.
#
# The old pipeline discovered monthly VolumeTrend sitting inside daily subtheme
# 2_8, and weekly claims inside monthly 12_1, then patched both afterwards.
# Recording cadence per factor makes one-cadence-per-subtheme an assert in
# notebook 02 rather than a patch.

FULL_MOM = {'daily':   'agg_market_daily_full_moments',
            'monthly': 'agg_market_monthly_full_moments'}
MEANS    = {'daily':   'agg_market_daily_means',
            'weekly':  'weekly_raw',
            'monthly': 'agg_market_monthly_means'}
SKIP = {'date', 'target_daily_return', 'target_monthly_return'}

stock_base = {}          # base factor -> cadence, for stock-level factors
for freq, tag in FULL_MOM.items():
    cols = pd.read_parquet(UNI_DIR / f'{tag}.parquet').columns
    for b in {c[:-len('_cwmean')] for c in cols if c.endswith('_cwmean')}:
        stock_base[b] = freq

cadence, level, conflicts = {}, {}, []
for freq, tag in MEANS.items():
    cols = pd.read_parquet(UNI_DIR / f'{tag}.parquet').columns
    bmap = rv.base_factor_map(cols)
    for c in cols:
        if c in SKIP:
            continue
        b = bmap[c]
        if b in cadence and cadence[b] != freq:
            conflicts.append((b, cadence[b], freq))
        cadence[b] = freq
        level[b] = 'stock' if b in stock_base else 'macro'

print('=' * 100)
print('CADENCE AND LEVEL')
print('=' * 100)
print(f'  base factors catalogued : {len(cadence)}')
print(f'  cadence  : ' + '  '.join(
    f'{k} {sum(v == k for v in cadence.values())}'
    for k in ('daily', 'weekly', 'monthly')))
print(f'  level    : ' + '  '.join(
    f'{k} {sum(v == k for v in level.values())}' for k in ('stock', 'macro')))
if conflicts:
    print(f'\n  !! {len(conflicts)} factor(s) appear at more than one cadence:')
    for b, a, c in conflicts:
        print(f'     {b}: {a} and {c}')

# ── BASE FACTORS THAT NEED A THEME ───────────────────────────────────────────
print('\n' + '=' * 100)
print('BASE FACTORS IN THE FINAL DATASETS')
print('=' * 100)

present = {}
for name, meta in FINAL_META.items():
    cols = pd.read_parquet(ASM_DIR / f'{name}.parquet').columns
    bmap = rv.base_factor_map(cols)
    present[name] = {bmap[c] for c in cols if c not in meta}
    print(f'  {name:<18} {len(cols):>5} columns   {len(present[name]):>4} base factors')

new_base = set().union(*present.values())
print(f'\n  union across all three: {len(new_base)}')

uncat = sorted(new_base - set(cadence))
if uncat:
    print(f'  !! {len(uncat)} present in an assembled table but not in any source: {uncat}')

# The union step enforced an identical stock-level base factor set, so the panel
# should be a clean subset of agg_means -- panel = stock + macro, aggregate adds
# only moments, which are not base factors.
print(f'  panel not in agg_means : {sorted(present["panel"] - present["agg_means"])}')
print(f'  agg_means not in panel : {sorted(present["agg_means"] - present["panel"])}')

CADENCE AND LEVEL
  base factors catalogued : 579
  cadence  : daily 290  weekly 32  monthly 257
  level    : stock 288  macro 291

BASE FACTORS IN THE FINAL DATASETS
  agg_means            581 columns    579 base factors
  agg_full_moments    1706 columns    579 base factors
  panel                583 columns    579 base factors

  union across all three: 579
  panel not in agg_means : []
  agg_means not in panel : []


In [2]:
# ── THE SAME NAME, THREE DIFFERENT QUANTITIES ────────────────────────────────
# `Tax` in the panel is one stock's value. `Tax` in agg_means is the
# cap-weighted market average. `Tax_cwmean` in agg_full_moments is the same
# quantity as the second. This is not a bug -- it is why notebook 03's collision
# assert fired when the panel first tried to take the aggregate's means, and why
# the panel takes macro only.
#
# It matters for the theme file: one assignment keyed on BASE factor serves all
# three datasets, and the `level` column records which reading applies.

print('=' * 100)
print('NAME REUSE ACROSS DATASETS')
print('=' * 100)

shared = sorted(present['panel'] & present['agg_means'])
stock_shared = [b for b in shared if level.get(b) == 'stock']
macro_shared = [b for b in shared if level.get(b) == 'macro']

print(f'  names in both panel and agg_means : {len(shared)}')
print(f'    stock-level ({len(stock_shared):>3}) -- panel: this stock; '
      f'agg_means: cap-weighted market mean')
print(f'    macro       ({len(macro_shared):>3}) -- identical value in both')
print(f'\n  examples (stock): {stock_shared[:8]}')
print(f'  examples (macro): {macro_shared[:8]}')

# Every stock base factor in the means table must have all five moments in the
# full-moments table, or the base_factor mapping is inconsistent between them.
fm_cols = set(pd.read_parquet(UNI_DIR / 'agg_market_daily_full_moments.parquet').columns)
fm_cols |= set(pd.read_parquet(UNI_DIR / 'agg_market_monthly_full_moments.parquet').columns)
partial = {b: [s for s in MOMENTS if f'{b}{s}' not in fm_cols]
           for b in stock_base if any(f'{b}{s}' not in fm_cols for s in MOMENTS)}
print(f'\n  stock factors missing some moments: {len(partial)}')
for b, miss in sorted(partial.items())[:12]:
    print(f'    {b:<32} missing {miss}')
print('  (expected: Stage 2 drop_undefined removed some skew/kurt columns, and')
print('   R2/R6 removed individual moments from ~28 factors whose cwmean survived)')

NAME REUSE ACROSS DATASETS
  names in both panel and agg_means : 579
    stock-level (288) -- panel: this stock; agg_means: cap-weighted market mean
    macro       (291) -- identical value in both

  examples (stock): ['AM', 'AbnormalAccruals', 'Accruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'Beta', 'BetaFP']
  examples (macro): ['a_oas', 'aa_oas', 'aaa_oas', 'aluminum_monthly', 'am_long', 'am_net', 'am_net_chg', 'am_net_pct']

  stock factors missing some moments: 21
    ChTax                            missing ['_cwkurt']
    ConvDebt                         missing ['_spread']
    CredRatDG                        missing ['_spread']
    DebtIssuance                     missing ['_spread']
    DelBreadth                       missing ['_cwstd', '_cwkurt']
    EquityDuration                   missing ['_cwstd', '_cwkurt']
    IntanCFP                         missing ['_cwkurt']
    PctAcc                           missing ['_cwstd']
    PctTotAcc                        miss

In [5]:
# ── THE OLD ASSIGNMENT: WHICH FILE, AND HOW TO NORMALISE ─────────────────────
#
# Read from Data/Splits/themes/, NOT Stage_3_Model_Ready/themes/.
#
# The Stage_3 copy has the float-truncation bug: subtheme_id '1.10' was written
# to CSV, pandas read it back as float 1.1, and it collided with the real '1.1'.
# Six subthemes were affected -- 1.10, 2.10, 3.10, 8.10, 9.10, 12.10 -- which is
# why a 107-subtheme mapping reads back as 101. The Splits copy has the fix
# applied and already uses underscore IDs; it also has the 2_11 and 12_11
# cadence splits, so both known mixed-cadence conflicts should come back clean.
#
# NORMALISATION: strip the monthly_ prefix ONLY.
#
# The old file is the MEANS assignment, so every name is already a bare base
# factor -- there are no moment suffixes in it. Stripping _spread would corrupt
# thirteen factors whose names genuinely end in _spread: bbb_aaa_spread,
# bb_bbb_spread, ff_2y_spread, ff_10y_spread, brent_wti_spread, vix_term_spread,
# bid_ask_spread, rec_buy_sell_spread, bull_bear_spread, am_spread, lev_spread,
# dealer_spread, other_spread. Each would appear as both "new" and "dropped".

OLD = Path('../../../Data/Splits/themes/combined_means_theme_assignment.csv')


def norm(c):
    """Old column name -> new base factor name."""
    return c[len('monthly_'):] if c.startswith('monthly_') else c


old = pd.read_csv(OLD)
# No-op on the corrected file; kept as a guard in case the source ever changes.
old['subtheme_id'] = old['subtheme_id'].astype(str).str.replace('.', '_', regex=False)
old['norm'] = old['column'].map(norm)

print('=' * 100)
print('MATCH AGAINST THE OLD ASSIGNMENT')
print('=' * 100)
print(f'  source: {OLD}')
print(f'  {len(old)} rows, {old["norm"].nunique()} base factors, '
      f'{old["subtheme_id"].nunique()} subthemes')

n_sub = old['subtheme_id'].nunique()
if n_sub < 105:
    print(f'\n  !! only {n_sub} subthemes -- expected ~107. The float-truncation')
    print(f'     bug may still be present. Check for X_1 subthemes that have')
    print(f'     absorbed their X_10 sibling.')

clash = old.groupby('norm')['subtheme_id'].nunique().loc[lambda s: s > 1]
if len(clash):
    print(f'\n  !! {len(clash)} names collide after stripping monthly_:')
    for n in clash.index:
        print(old[old['norm'] == n][['column', 'subtheme_id', 'subtheme_name']]
              .to_string(index=False))
else:
    print('  no collisions from stripping the monthly_ prefix')

old_map = (old.drop_duplicates('norm')
              .set_index('norm')[['theme_id', 'theme_name',
                                  'subtheme_id', 'subtheme_name']]
              .to_dict('index'))

matched = sorted(new_base & set(old_map))
missing = sorted(new_base - set(old_map))
gone    = sorted(set(old_map) - new_base)

print(f'\n  matched  {len(matched):>4}   inherit their old theme')
print(f'  NEW      {len(missing):>4}   need classifying by hand')
print(f'  dropped  {len(gone):>4}   classified before, absent now')

# ── A NEW FACTOR THAT LOOKS LIKE AN OLD ONE ──────────────────────────────────
# Catches any remaining normalisation mismatch: if a "new" factor's name is a
# near-match for a "dropped" one, it is probably the same factor under a changed
# name rather than something genuinely new.

near = []
for m in missing:
    for g in gone:
        if m.startswith(g) or g.startswith(m):
            near.append((m, g, old_map[g]['subtheme_id'],
                         old_map[g]['subtheme_name']))
if near:
    print(f'\n  !! {len(near)} "new" factor(s) resemble a "dropped" one:')
    for m, g, sid, sname in near:
        print(f'     {m:<28} ~ {g:<24} ({sid} {sname})')
    print('     If these are the same factor, the normalisation is still wrong.')

print('\n' + '=' * 100)
print(f'NEW FACTORS -- {len(missing)} needing classification')
print('=' * 100)
if missing:
    print(pd.DataFrame({
        'base_factor': missing,
        'cadence':     [cadence.get(b, '?') for b in missing],
        'level':       [level.get(b, '?') for b in missing],
    }).sort_values(['cadence', 'level', 'base_factor']).to_string(index=False))
else:
    print('  none')

print('\n' + '=' * 100)
print(f'DROPPED -- {len(gone)}')
print('=' * 100)
print('Expected: 85 union drops, 12 high-imputation, 8 calendar (14_1),')
print('5 Fed levels replaced by weekly _diff versions, plus Stage 1.5/2')
print('exclusions that never reached the new pipeline.\n')
print(old[old['norm'].isin(gone)].drop_duplicates('norm')
         .groupby(['subtheme_id', 'subtheme_name'])
         .agg(n=('norm', 'size'),
              examples=('norm', lambda s: ', '.join(sorted(s)[:4])))
         .reset_index().sort_values('n', ascending=False).to_string(index=False))

emptied = [(sid, g['subtheme_name'].iloc[0], len(g))
           for sid, g in old.drop_duplicates('norm').groupby('subtheme_id')
           if not (set(g['norm']) & new_base)]
if emptied:
    print(f'\n  subthemes with no surviving members ({len(emptied)}):')
    for sid, sname, n in sorted(emptied):
        print(f'    {sid:<8} {sname:<45} was {n} factors')
    print('\n  14_1 Calendar is a genuine deletion -- all 8 features were')
    print('  excluded in Stage 3. 9_7 Fed Balance Sheet is a RENAME: its five')
    print('  levels were replaced by weekly _diff versions in the Stage 3')
    print('  rebuild, four of which survive.')

MATCH AGAINST THE OLD ASSIGNMENT
  source: ..\..\..\Data\Splits\themes\combined_means_theme_assignment.csv
  714 rows, 714 base factors, 107 subthemes
  no collisions from stripping the monthly_ prefix

  matched   574   inherit their old theme
  NEW         5   need classifying by hand
  dropped   140   classified before, absent now

  !! 5 "new" factor(s) resemble a "dropped" one:
     bank_credit_diff             ~ bank_credit              (9_7 Fed Balance Sheet & Banking)
     ci_loans_diff                ~ ci_loans                 (9_7 Fed Balance Sheet & Banking)
     fed_assets_diff              ~ fed_assets               (9_7 Fed Balance Sheet & Banking)
     open_interest_diff           ~ open_interest            (3_12 CFTC VIX Futures Positioning)
     reserves_diff                ~ reserves                 (9_7 Fed Balance Sheet & Banking)
     If these are the same factor, the normalisation is still wrong.

NEW FACTORS -- 5 needing classification
       base_factor cadence 

In [7]:
starter = pd.DataFrame([{
    'base_factor':   b,
    'cadence':       cadence.get(b, '?'),
    'level':         level.get(b, '?'),
    'theme_id':      old_map.get(b, {}).get('theme_id', ''),
    'theme_name':    old_map.get(b, {}).get('theme_name', ''),
    'subtheme_id':   old_map.get(b, {}).get('subtheme_id', ''),
    'subtheme_name': old_map.get(b, {}).get('subtheme_name', ''),
    'status':        'matched' if b in old_map else 'NEEDS CLASSIFYING',
} for b in sorted(new_base)])

starter.to_csv(OUT_DIR / 'starter_assignment.csv', index=False)
print(f'  saved -> starter_assignment.csv  ({len(starter)} rows)')

m = starter[starter['status'] == 'matched']

# ── ONE CADENCE PER SUBTHEME ─────────────────────────────────────────────────
# Required by the sparse KAN's update mask: a subtheme fires on the days its
# features change, so one mixing daily and monthly has no single update day.
# The corrected source file already carries the 2_11 (monthly VolumeTrend out of
# daily 2_8) and 12_11 (weekly claims out of monthly 12_1) splits, so this
# should come back clean. Anything here is a NEW conflict.

print('\n' + '=' * 100)
print('SUBTHEMES WITH MIXED CADENCE  (must be split in notebook 02)')
print('=' * 100)

mix = (m.groupby(['subtheme_id', 'subtheme_name'])['cadence'].nunique()
         .loc[lambda s: s > 1].index.tolist())
if mix:
    for sid, sname in mix:
        sub = m[m['subtheme_id'] == sid].sort_values(['cadence', 'base_factor'])
        print(f'\n  {sid:<8} {sname}   {dict(sub["cadence"].value_counts())}')
        print(sub[['base_factor', 'cadence', 'level']].to_string(index=False))
else:
    print('  none -- the 2_11 and 12_11 splits in the source file hold')

print('\n' + '=' * 100)
print('SUBTHEMES MIXING STOCK AND MACRO  (informational, not a problem)')
print('=' * 100)
lm = (m.groupby(['subtheme_id', 'subtheme_name'])['level'].nunique()
        .loc[lambda s: s > 1].index.tolist())
for sid, sname in lm:
    sub = m[m['subtheme_id'] == sid]
    print(f'  {sid:<8} {sname:<45} {dict(sub["level"].value_counts())}')
if not lm:
    print('  none')

# ── SIZES, AND WHAT TO MERGE ─────────────────────────────────────────────────
# A one-feature subtheme is a passthrough edge, not a group -- it gives the
# sparse KAN nothing the dense model lacks. Target is 3+, with 2 tolerated where
# no sensible sibling exists.
#
# Merge candidates are listed WITHIN theme and WITHIN cadence, because a
# subtheme's cadence sets its update day and a daily orphan cannot join a
# monthly subtheme.

print('\n' + '=' * 100)
print('SUBTHEME SIZES')
print('=' * 100)

sizes = (m.groupby(['theme_id', 'theme_name', 'subtheme_id', 'subtheme_name'])
           .agg(n=('base_factor', 'size'),
                cadence=('cadence', lambda s: s.mode()[0]))
           .reset_index())

print(f'  {len(sizes)} subthemes   sizes {sizes["n"].min()}-{sizes["n"].max()}   '
      f'median {sizes["n"].median():.0f}')
for k in (1, 2, 3):
    print(f'  with {k} member(s): {int((sizes["n"] == k).sum())}')

small = sizes[sizes['n'] <= 2].sort_values(['theme_id', 'subtheme_id'])
if len(small):
    print(f'\n  {len(small)} subtheme(s) below 3, with their same-theme, '
          f'same-cadence siblings:\n')
    for _, r in small.iterrows():
        sib = sizes[(sizes['theme_id'] == r['theme_id']) &
                    (sizes['cadence'] == r['cadence']) &
                    (sizes['subtheme_id'] != r['subtheme_id'])]
        members = sorted(m.loc[m['subtheme_id'] == r['subtheme_id'], 'base_factor'])
        print(f'  {r["subtheme_id"]:<7} {r["subtheme_name"]:<42} '
              f'n={r["n"]}  {r["cadence"]}')
        print(f'          members: {", ".join(members)}')
        if len(sib):
            opts = '  |  '.join(f'{s.subtheme_id} {s.subtheme_name} (n={s.n})'
                                for s in sib.itertuples())
            print(f'          merge into: {opts}')
        else:
            print(f'          no same-theme same-cadence sibling -- keep or '
                  f'move to another theme')
        print()

print('=' * 100)
print('PER-THEME SUMMARY')
print('=' * 100)
print(sizes.groupby(['theme_id', 'theme_name'])
           .agg(subthemes=('subtheme_id', 'size'),
                factors=('n', 'sum'),
                min_size=('n', 'min'))
           .reset_index().to_string(index=False))

  saved -> starter_assignment.csv  (579 rows)

SUBTHEMES WITH MIXED CADENCE  (must be split in notebook 02)
  none -- the 2_11 and 12_11 splits in the source file hold

SUBTHEMES MIXING STOCK AND MACRO  (informational, not a problem)
  1_10     Monthly Liquidity                             {'stock': np.int64(3), 'macro': np.int64(2)}
  3_11     CBOE SKEW                                     {'macro': np.int64(4), 'stock': np.int64(1)}

SUBTHEME SIZES
  106 subthemes   sizes 1-20   median 5
  with 1 member(s): 7
  with 2 member(s): 10
  with 3 member(s): 14

  17 subtheme(s) below 3, with their same-theme, same-cadence siblings:

  1_2     TAQ Effective Spreads                      n=2  daily
          members: effectivespread_percent_ave, effspread_pct_rel_5d
          merge into: 1_1 CRSP Bid-Ask Spread & Dynamics (n=5)  |  1_3 Quoted Spreads & Intraday NBBO (n=4)  |  1_4 Best-Level Depth (n=2)  |  1_5 Intraday Depth Profile (n=7)  |  1_6 Depth Imbalance & Dynamics (n=5)  |  1_7 Price 

In [8]:
import pyarrow.parquet as pq
from pathlib import Path

old = pq.read_schema(Path('../../../Data/Splits/Split_A/full_moments_test.parquet')).names
new = pq.read_schema(Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready'
                          '/04_splits/Split_A/agg_full_moments_test.parquet')).names

# old meta: date, target_daily_return, minret_5d, minret_5d_z, y_binary
# new meta: date, target_daily_return, minret_5d_pct, y_binary
print(f'  old  {len(old):>5} cols   {len(old) - 5:>5} features')
print(f'  new  {len(new):>5} cols   {len(new) - 4:>5} features')

  old   2209 cols    2204 features
  new   1708 cols    1704 features
